In [11]:
import numpy as np
import pandas as pd

In [12]:
df = pd.read_csv("/content/glass.csv")



In [13]:
df.shape

(214, 10)

In [14]:
df.columns

Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')

In [15]:
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


Observations
- "Type" is the output column
- Yes all columns are numeric
- No Id column is there

In [16]:
df["y"] = (df["Type"]==1).astype(int)

df = df.drop(columns=["Type"])
df.head(10)

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,y
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.00,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.00,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.00,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.00,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.00,1
5,1.51596,12.79,3.61,1.62,72.97,0.64,8.07,0.0,0.26,1
6,1.51743,13.30,3.60,1.14,73.09,0.58,8.17,0.0,0.00,1
7,1.51756,13.15,3.61,1.05,73.24,0.57,8.24,0.0,0.00,1
8,1.51918,14.04,3.58,1.37,72.08,0.56,8.30,0.0,0.00,1
9,1.51755,13.00,3.60,1.36,72.99,0.57,8.40,0.0,0.11,1


We converted the problem into Binary Classification.
- Type 1 -> 1
- All others ->0

Basically simplifying

In [18]:
X = df.drop(columns=["y"]).values
y = df["y"].values


We are separating features(x) and labels(y).

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



- Splitting the data here into training and testing.
- Model learns on training data and evaluated on testing data.

In [22]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


We scale the features so that all input values are in a similar range.If we do not scale, features with large values can dominate learning.

In [23]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


The sigmoid function converts the linear score
into a probability between 0 and 1.

Unlike the step function, it provides smooth output.


In [24]:
def predict_proba(X, w, b):
    z = X @ w + b
    p = sigmoid(z)
    return p


We are computing z and applying sigmoid to get probability.

In [25]:
def loss(y, p):
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))


It penalizes confident wrong predictions more.



In [26]:
def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y

    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)

    return w, b


We update weights using gradient descent.

(p - y) tells direction and strength of correction.

Learning rate controls how big each update step is.


In [27]:
w = np.zeros(X_train.shape[1])
b = 0.0

lr = 0.1
epochs = 100

for epoch in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)


We train the model for multiple epochs.

Each epoch adjusts weights to reduce error.
Over time, predictions become more accurate.


In [28]:
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

p_test = predict_proba(X_test, w, b)

y_pred_05 = predict_label(p_test, 0.5)
y_pred_07 = predict_label(p_test, 0.7)

print("Accuracy (0.5):", np.mean(y_pred_05 == y_test))
print("Accuracy (0.7):", np.mean(y_pred_07 == y_test))


Accuracy (0.5): 0.8604651162790697
Accuracy (0.7): 0.7209302325581395


The model outputs probabilities.

We apply a threshold to convert them into class labels.

When we increase the threshold, the model becomes more strict.
It predicts positive class only when probability is very high.

A higher threshold can be safer in quality control,
because it reduces the risk of accepting incorrect glass.

Logistic regression differs from perceptron because perceptron
makes hard decisions using a step function,
while logistic regression outputs probabilities using the sigmoid function.

The sigmoid function is important because it provides smooth,
interpretable confidence values and avoids abrupt decision flips near the boundary.

However, one limitation still remains: logistic regression
can only learn linear decision boundaries.
It cannot model complex nonlinear relationships without feature transformation.
